In [1]:
import os
import random
import pandas as pd
import json

# Mount to google drive
# from google.colab import drive
# drive.mount('/content/drive')

# Change it to your google drive path where this notebook located.
drive_path = '/Users/samyiin/Projects/ZipfLawAnalysis'
os.chdir(drive_path)

# Description
This notebook will take the user's "hand written" locations, and map them into one of the Countries (what about controversial ones?). 

In [2]:
from Utils.GeopyAPI import map_location_to_country
import pandas as pd

# load the users
df_users = pd.read_csv("Database/TempData/GatherData/Filter_1_Parallel/Results/users_filter_1.csv")
# get all the unique locations
unique_locations = df_users['location'].unique()
unique_locations = [location for location in unique_locations if pd.notna(location)]


In [3]:
len(unique_locations)

30285

In [4]:
from joblib import Memory
# Work around for joblib caching in jupyter notebook "joblib persistence across sessions/machines"
def cache(mem, module, **mem_kwargs):
    # model is the notebook/python file name: Jupyter notebook's name is always changing so we need this work around
    def cache_(f):
        f.__module__ = module
        f.__qualname__ = f.__name__
        return mem.cache(f, **mem_kwargs)

    # return the cache function that will always create same name for cache directory
    return cache_
# Create a memory object with a cache directory
memory = Memory(location="FunctionCache", verbose=0)
cache_dir = 'map_location_to_country'
@cache(memory, cache_dir)
def cached_map_location_to_country(location):
    # for the rate limit
    time.sleep(0.9)
    return map_location_to_country(location)

# Store this
def store_or_read(file_path, data=None):
    """
    Store a list to a file if it doesn't exist, or read it if it exists.
    
    Parameters:
        file_path (str): Path to the JSON file.
        data (list): List to store if the file doesn't exist. Defaults to None.
        
    Returns:
        list: The list read from the file or the data written to it.
    """
    if os.path.exists(file_path):
        # File exists, read the list
        with open(file_path, 'r') as file:
            stored_data = json.load(file)
        print("File exists. Data read from file.")
        return stored_data
    else:
        # File doesn't exist, write the provided data
        if data is None:
            raise ValueError("File does not exist, and no data provided to write.")
        with open(file_path, 'w') as file:
            json.dump(data, file)
        print("File does not exist. Data written to file.")
        return data


# GeoPy

In [76]:
import time
from tqdm import tqdm

# create map from location to countries
dic_geo_location_to_country = {}
for location in tqdm(unique_locations):
    try:
        dic_geo_location_to_country[location] = cached_map_location_to_country(location)
    except:
        # there are some weird ones that goepy somehow just cannot map??? like "US o A"
        dic_geo_location_to_country[location] = 'Unknown'



100%|████████████████████████████████████████████████████████████████████| 30285/30285 [01:08<00:00, 442.74it/s]


In [79]:
dic_geo_location_to_country_file_path = "Database/MapLocationToCountry/dic_geo_location_to_country.json"
dic_geo_location_to_country = store_or_read(dic_geo_location_to_country_file_path, data=dic_geo_location_to_country)


dic_unidentifiable_locations = {k: v for k, v in dic_geo_location_to_country.items() if v in ['Unknown', 'Ambiguous']}
df_temp = pd.DataFrame(list(dic_unidentifiable_locations.items()), columns=["Location", "Country"])
df_temp.to_csv("Database/MapLocationToCountry/geopy_unidentifiable.csv", index=False)

dic_identifiable_locations = {k: v for k, v in dic_geo_location_to_country.items() if v not in ['Unknown', 'Ambiguous']}
df_temp = pd.DataFrame(list(dic_identifiable_locations.items()), columns=["Location", "Country"])
df_temp.to_csv("Database/MapLocationToCountry/geopy_identifiable.csv", index=False)

File does not exist. Data written to file.


# LLMs

In [8]:
'''
separate these functions and objects, make it from outer scope, so that if we change them it won't affect the cache
'''

from Utils.GeminiAPI import GeminiChatBot
import json
import re
from tqdm import tqdm

'''
We tried to use Gemini, but it is really just hard to use...
'''

# Use chatgpt-4o for precise actions
from Utils.ChatGPTAPI import GPTChatBot
from Utils.Secrete import SecreteLoader

chatgpt_api_key = SecreteLoader().get_openai_api_key('Sam')


chatbot_request_body = {
    "model": "gpt-4o",
    "messages": [{"role": "system", "content":"You are a helpful assistant."}],
    "temperature": 0.7,
}
chat_bot = GPTChatBot(chatgpt_api_key, chatbot_request_body)

interpreter_prompt = """Please convert the input text into a json dictionary. The dictionary should map address to a country name or "Unknown"/"Ambiguous". You output will be directly feed into json.loads(), so your output should only contain the dictionary, starting with '{' and ending with '}'. If there are any spelling errors you keep the spelling errors in the keys, make sure the keys are the same as the input."""

request_body = {
    "model": "gpt-4o",
    "messages": [{"role": "system", "content":interpreter_prompt}],
    "temperature": 0.7,
}
interpreter = GPTChatBot(chatgpt_api_key, request_body)


def json_read_response_to_dic(refined_answer_response):
    try: 
        # compile response back to json
        dic_response = json.loads(refined_answer_response)
    except:
        # make sure it's json safe
        refined_answer_response = re.sub(r'\\[^\\"bfnrt]', '', refined_answer_response)
        dic_response = json.loads(refined_answer_response)

    return dic_response

def get_unanswered_locations(list_unknown_locations, dic_response):
    # Normalize strings: remove extra spaces and convert to lowercase for comparison
    def normalize_string(s):
        # Remove punctuation and normalize spaces
        s = re.sub(r'[.,]', '', s)  # Remove commas and dots
        s = re.sub(r'\s+', '', s)  # Normalize multiple spaces
        return s.strip().lower()  # Strip leading/trailing spaces and convert to lowercase

    
    # Create a mapping of normalized location names to the original list names
    normalized_map = {normalize_string(loc): loc for loc in list_unknown_locations}
    
    # Create a new dictionary with corrected keys
    corrected_dict = {}
    for key, value in dic_response.items():
        normalized_key = normalize_string(key)
        if normalized_key in normalized_map:
            corrected_dict[normalized_map[normalized_key]] = value
        else:
            corrected_dict[key] = value  # Keep unmatched keys as they are
    dic_response = corrected_dict
    
    # Check the keys are correct
    dic_response_keys_set = set(dic_response.keys())
    list_set = set(list_unknown_locations)
    
    # Find differences
    keys_only_in_dict = dic_response_keys_set - list_set
    keys_only_in_list = list_set - dic_response_keys_set
    
    # Display differences
    if keys_only_in_dict or keys_only_in_list:
        print("The dictionary keys and the list are not identical.")
        if keys_only_in_dict:
            print(f"Keys in the dictionary but not in the list: {keys_only_in_dict}")
            # delete these useless keys
            for key in keys_only_in_dict: dic_response.pop(key)
        if keys_only_in_list:
            print(f"Keys in the list but not in the dictionary: {keys_only_in_list}")
    else:
        print("The dictionary keys and the list of strings are identical.")
    return keys_only_in_list

'''
The cached function: Don't touch it!
it requires these things from outer scope:
1. function: json_read_response_to_dic
2. function: get_unanswered_locations
3. object: chat_bot
4. object: interpreter
One thing we learned: when making cache function, make it abstract so that when we change implementation we don't ruin everything. 
'''
cache_dir = 'llm_map_list_location_to_country'
@cache(memory, cache_dir)
def LLM_solve_unknown_locations(list_unknown_locations):
    """
    Return the dictionary that map location to countries
    Return not successful ones
    """
    prompt = f"I have a list of location here: {list_unknown_locations}, I need you to help identify what are their countries. It's possible that there are unidentifiable/invalid/fictional locations, in that case return Unknown. If there are more than one countries, return two countries separated by &. If there are any spelling errors you keep the spelling errors, make sure the keys are the same as the input "
    # Use chat_bot to give an sematic answer for the location
    raw_answer_response = chat_bot.chat(prompt)
    prompt = raw_answer_response
    
    # interpreter compiles the raw response from chat_bot
    refined_answer_response = interpreter.chat(prompt)
    
    # decypher the answer from interpreter, this can be tricky given the LLM's answer.
    dic_response = json_read_response_to_dic(refined_answer_response)
    
    # Match the answer of LLMs to the list of location, find the ones LLM forgots to answer
    unanswered_locations = get_unanswered_locations(list_unknown_locations, dic_response)
    return dic_response, unanswered_locations



In [9]:
dic_geo_location_to_country_file_path = "Database/MapLocationToCountry/dic_geo_location_to_country.json"
dic_geo_location_to_country = store_or_read(dic_geo_location_to_country_file_path, data=None)

        
dic_geo_unidentifiable_locations = {k: v for k, v in dic_geo_location_to_country.items() if v in ['Unknown', 'Ambiguous']}
list_geo_unidentifiable_locations = list(dic_geo_unidentifiable_locations.keys())

list_geo_unidentifiable_locations_file_path = "FunctionCache/joblib/llm_map_list_location_to_country/LLM_solve_unknown_locations/list_geo_unidentifiable_locations.json"
list_geo_unidentifiable_locations = store_or_read(list_geo_unidentifiable_locations_file_path, data=list_geo_unidentifiable_locations)

# Bulk process the locations: each time 100
final_dic = {}
failed_attempt = []
list_geo_unidentifiable_locations = list_geo_unidentifiable_locations
for i in tqdm(range(0, len(list_geo_unidentifiable_locations), 100)):
    list_unknown_locations = list_geo_unidentifiable_locations[i:i + 100]
    success = False
    while not success:
        try:
            dic_response, keys_only_in_list = LLM_solve_unknown_locations(list_unknown_locations)
            success = True
        except:
            print(f'retry{i}')
            pass
    final_dic.update(dic_response)
    failed_attempt += keys_only_in_list


File exists. Data read from file.
File exists. Data read from file.


100%|█████████████████████████████████████████████████████████████████████████████████| 45/45 [00:00<00:00, 1150.66it/s]


In [10]:
failed_attempt

['San Francisco',
 'Champiagn, USA',
 'london',
 ' ┣▇ᶠᶸᶜᵏᵧₒᵤ▇▇▇▇═──ด้้้้้็็็็็้้้้้็็็็็้้้้้้้้็็็็็้้้้้็็็็็้้้้้้้้็็็็็้้้้้็็็็็้้้้้้้้็็็็\xa0͇͇͇͇͇͇͇͇͇͇͇͇͇͇͇͇͇͇͇͇͇͇͇͇͇\xa0ͤͤͤͤ ͬͬͬͬ ͬͬͬͬ ͦͦͦͦ ͬͬ  DOGE: DGLiTcH1iTV5zMT2WK69siiocZfKyFN2h1',
 'San  Francisco',
 'Palestine',
 'NEW DELHI',
 'lagos ',
 'Right here.',
 'NoWhere',
 'http://g.com/#\'"/onmouseover="prompt(1)"/x=',
 'CHINA BEIJING']

In [11]:
# rerun meaningful errors
dic_response, keys_only_in_list = LLM_solve_unknown_locations(['San Francisco', 'Champiagn, USA', 'london','San  Francisco', 'Palestine','NEW DELHI','lagos ','CHINA BEIJING'])
final_dic.update(dic_response)
dic_response, keys_only_in_list = LLM_solve_unknown_locations(['San Francisco'])
final_dic.update(dic_response)
dic_LLM_location_to_country = final_dic

In [12]:
# store the dic
dic_LLM_location_to_country_file_path = "Database/MapLocationToCountry/dic_LLM_location_to_country.json"
dic_LLM_location_to_country = store_or_read(dic_LLM_location_to_country_file_path, data=dic_LLM_location_to_country)

# save the LLM unidentifiable ones
dic_unidentifiable_locations = {k: v for k, v in dic_LLM_location_to_country.items() if v in ['Unknown', 'Ambiguous', 'Fictional']}
df_temp = pd.DataFrame(list(dic_unidentifiable_locations.items()), columns=["Location", "Country"])
df_temp.to_csv("Database/MapLocationToCountry/LLM_unidentifiable.csv", index=False)

# save the LLM identifiable ones
dic_identifiable_locations = {k: v for k, v in dic_LLM_location_to_country.items() if v not in ['Unknown', 'Ambiguous', 'Fictional']}
df_temp = pd.DataFrame(list(dic_identifiable_locations.items()), columns=["Location", "Country"])
df_temp.to_csv("Database/MapLocationToCountry/LLM_identifiable.csv", index=False)

File exists. Data read from file.


## LLM Post Process
The results of LLM still need some refinement, like turn USA back to United States, and some contains two or more countries, we will separate them by & sign, and some are "USA or Argentina", we will change it to ambiguous. 

In [13]:
dic_geo_location_to_country_file_path = "Database/MapLocationToCountry/dic_geo_location_to_country.json"
dic_geo_location_to_country = store_or_read(dic_geo_location_to_country_file_path, data=None)

dic_LLM_location_to_country_file_path = "Database/MapLocationToCountry/dic_LLM_location_to_country.json"
dic_LLM_location_to_country = store_or_read(dic_LLM_location_to_country_file_path, data=None)


File exists. Data read from file.
File exists. Data read from file.


In [14]:
# Manually handle the rest of the cases: There aren't a lot
dic_manually_handle = {'Latin America': "Ambiguous", 
                       'Palestine': "Palestinian Territory", 
                       'USA or Argentina': "Ambiguous", 
                       'USA': "United States", 
                       'European Union': "Ambiguous", 
                       'Fictional': "Fictional", 
                       'United Arab Emirates/Oman': "United Arab Emirates", 
                       'Colombia or Spain (Ambiguous without more context)': "Ambiguous", 
                       'Ukraine/Russia (disputed)': "Ukraine", 
                       'Tuvalu':'Tuvalu', 
                       'England': "United Kingdom", 
                       'Czech Republic':'Czechia', 
                       'UAE': "United Arab Emirates", 
                       'Puerto Rico or Argentina': "Ambiguous", 
                       'Korea':"South Korea", 
                       'CIS (Commonwealth of Independent States)':"Ambiguous" , 
                       'Puerto Rico': "United States", 
                       'Wales': "United Kingdom", 
                       'Macau': "China", 
                       'Southeast Asia': "Ambiguous", 
                       'UK': "United Kingdom", 
                       'Hong Kong': "China", 
                       'Macedonia': "North Macedonia", 
                       'French Polynesia': "France", 
                       'Canada or UK': "Ambiguous", 
                       'Balkans (region)': "Ambiguous", 
                       'Spain or Colombia': "Ambiguous"}

In [15]:
for location, llm_country_name in dic_LLM_location_to_country.items():
    # handle the & case
    countries = llm_country_name.split('&')
    # Trim any extra whitespace from the split results
    countries = [country.strip() for country in countries]
    tem_countries_list = []
    for country in countries:
        if country in dic_manually_handle:
            tem_countries_list.append(dic_manually_handle[country])
        else:
            tem_countries_list.append(country)
    # if there is only one country this will not do anthing but add the country
    dic_LLM_location_to_country[location] = ' & '.join(tem_countries_list)

In [16]:
LLM_unique_values = set(dic_LLM_location_to_country.values())
LLM_unique_countries = set()
for llm_country_name in LLM_unique_values:
    countries = llm_country_name.split('&')
    countries = set([country.strip() for country in countries])
    LLM_unique_countries = LLM_unique_countries | countries
    
# unique values of geopy is already countries
geo_unique_countries = set(dic_geo_location_to_country.values())

# see difference
diff1 = LLM_unique_countries - geo_unique_countries
print("In LLM_unique_values but not in geo_unique_values:", diff1)

In LLM_unique_values but not in geo_unique_values: {'Fictional', 'Tuvalu'}


In [17]:
# store the dic
dic_LLM_location_to_country_file_path = "Database/MapLocationToCountry/dic_LLM_processed_location_to_country.json"
dic_LLM_location_to_country = store_or_read(dic_LLM_location_to_country_file_path, data=dic_LLM_location_to_country)

# save the LLM unidentifiable ones
dic_unidentifiable_locations = {k: v for k, v in dic_LLM_location_to_country.items() if v in ['Unknown', 'Ambiguous', 'Fictional']}
df_temp = pd.DataFrame(list(dic_unidentifiable_locations.items()), columns=["Location", "Country"])
df_temp.to_csv("Database/MapLocationToCountry/LLM_processed_unidentifiable.csv", index=False)

# save the LLM identifiable ones
dic_identifiable_locations = {k: v for k, v in dic_LLM_location_to_country.items() if v not in ['Unknown', 'Ambiguous', 'Fictional']}
df_temp = pd.DataFrame(list(dic_identifiable_locations.items()), columns=["Location", "Country"])
df_temp.to_csv("Database/MapLocationToCountry/LLM_processed_identifiable.csv", index=False)

File exists. Data read from file.


# Analysis for this method
1. Randomly sample 100 locations, see how accurate the geopy and combined method are.
2. If we have more time, randomly sample 100 locations and see how well LLM performs. 

In [18]:
# The Geopy Method
dic_geo_location_to_country_file_path = "Database/MapLocationToCountry/dic_geo_location_to_country.json"
dic_geo_location_to_country = store_or_read(dic_geo_location_to_country_file_path, data=None)

# The LLM Method, but only for unidentifiable of Geo
dic_LLM_location_to_country_file_path = "Database/MapLocationToCountry/dic_LLM_processed_location_to_country.json"
dic_LLM_location_to_country = store_or_read(dic_LLM_location_to_country_file_path, data=None)

# The hybrid method
dic_location_to_country = dic_geo_location_to_country.copy() 
dic_location_to_country.update(dic_LLM_location_to_country)
dic_location_to_country_file_path = "Database/MapLocationToCountry/dic_location_to_country.json"
dic_location_to_country = store_or_read(dic_location_to_country_file_path, data=dic_location_to_country)


File exists. Data read from file.
File exists. Data read from file.
File exists. Data read from file.


In [19]:
import random
k = 100
random.seed(42)
random_selection = random.sample(unique_locations, k)

In [20]:
dic_response, keys_only_in_list = LLM_solve_unknown_locations(random_selection)
keys_only_in_list

set()

In [22]:
dic_pure_LLM_location_to_country = dic_response
data = {
    'Selected': random_selection,
    'geopy_method': [dic_geo_location_to_country[item] for item in random_selection],
    'LLM_method': [dic_pure_LLM_location_to_country[item] for item in random_selection],
    'hybrid_method': [dic_location_to_country[item] for item in random_selection],
}

df_temp = pd.DataFrame(data)
df_temp.to_csv("Database/TempData/GatherData/Filter_2_DecypherLocation/Temp/MethodEvaluationResults.csv", index=False)

# Manually graded results

In [23]:
df_manual_graded = pd.read_csv("Database/TempData/GatherData/Filter_2_DecypherLocation/Temp/ManuallyGradedEvaluation.csv")

In [24]:
df_manual_graded['geo_accuracy'].value_counts()

geo_accuracy
 1    81
 0    10
-1     9
Name: count, dtype: int64

In [25]:
df_manual_graded['LLM_accuracy'].value_counts()

LLM_accuracy
 1    89
-1     9
 0     2
Name: count, dtype: int64

In [26]:
df_manual_graded['hybrid_accuracy'].value_counts()

hybrid_accuracy
 1    89
-1     9
 0     2
Name: count, dtype: int64

In [27]:
df_manual_graded['geo_unknown'].value_counts()

geo_unknown
TP    84
FN     8
FP     5
TN     3
Name: count, dtype: int64

In [28]:
df_manual_graded['LLM_unknown'].value_counts()

LLM_unknown
TP    90
TN     8
FN     2
Name: count, dtype: int64

In [29]:
df_manual_graded['hybrid_unknown'].value_counts()

hybrid_unknown
TP    92
TN     6
FP     2
Name: count, dtype: int64

In [30]:
df_manual_graded['geo_ambiguous'].value_counts()

geo_ambiguous
TN    96
FP     4
Name: count, dtype: int64

In [31]:
df_manual_graded['LLM_ambiguous'].value_counts()

LLM_ambiguous
TN    100
Name: count, dtype: int64

In [32]:
df_manual_graded['hybrid_ambiguous'].value_counts()

hybrid_ambiguous
TN    100
Name: count, dtype: int64